# Task 2 – Cleaning
**Who:** Joud & Maram (together)  
**Input:** `Food_Inspections.csv` (raw data from the Chicago Data Portal, in this project folder)  
**Output:** `food_clean.csv`, the clean file both feature notebooks start from.

The cleaning rules below are our **suggested** rules. If we change a rule after discussing, we change the cell and the note under it.
After each rule we print the number of rows, so we can see how much data each rule removes.

In [14]:
import pandas as pd

We only need pandas in this notebook (no plots).

In [15]:
food = pd.read_csv('Food_Inspections.csv')
food['Inspection Date'] = pd.to_datetime(food['Inspection Date'])
print(food.shape)

(317176, 17)


We read the raw file and turn the date text into a real date, so we can compare dates later.
**Rows at the start:** (317176, 17)

In [16]:
# Rule 1: keep only real inspection results
df = food[food['Results'].isin(['Pass', 'Pass w/ Conditions', 'Fail'])].copy()
print(df.shape)

(272287, 17)


**Rule 1.** Out of Business, No Entry, Not Ready and Business Not Located are not real inspections (nobody checked the food), so we remove them. We keep only Pass, Pass w/ Conditions and Fail.

In [17]:
# Rule 2: remove exact duplicate rows
df = df.drop_duplicates()
print(df.shape)

(272287, 17)


**Rule 2.** The dataset description warns that some rows are duplicated, two identical rows are one inspection, so we keep one.

In [18]:
# Rule 3: remove the same business inspected twice on the same date
df = df.drop_duplicates(subset=['License #', 'Inspection Date'])
print(df.shape)

(271223, 17)


**Rule 3.** If one business (same License #) has two rows on the same day, it is the same visit recorded twice, so we keep the first one.

In [19]:
# Rule 4: remove rows without a License #
df = df.dropna(subset=['License #'])
df = df[df['License #'] != 0]
print(df.shape)

(270584, 17)


**Rule 4.** We need the License # to follow each business over time (previous result, days since last inspection), rows with no License # or License # = 0 cannot be followed, so we remove them.

In [20]:
# Rule 5: fill missing text values
df['Facility Type'] = df['Facility Type'].fillna('Unknown')
df['Risk'] = df['Risk'].fillna('Unknown')
df['Risk'] = df['Risk'].replace('All', 'Unknown')
df['Risk'].value_counts()

Risk
Risk 1 (High)      206210
Risk 2 (Medium)     46806
Risk 3 (Low)        17549
Unknown                19
Name: count, dtype: int64

**Rule 5.** Facility Type and Risk are text columns with a few missing values, instead of deleting those rows we write "Unknown", and Risk = "All" is not a real risk level, so it also becomes "Unknown".

In [21]:
# Rule 6: keep the years we study
df['Year'] = df['Inspection Date'].dt.year
df = df[(df['Year'] >= 2015) & (df['Year'] <= 2022)]
print(df.shape)

(126787, 18)


**Rule 6.** We keep 2015–2022: about 3.5 years before the July 2018 change and 3.5 years after the transition year, so 2020 (COVID) isn’t the only “after” year we test.

In [22]:
# Target column: what the model will predict
def make_target(result):
    if result == 'Fail':
        return 1
    else:
        return 0

df['Fail'] = df['Results'].apply(make_target)
df['Fail'].value_counts()

Fail
0    98433
1    28354
Name: count, dtype: int64

**Target.** Fail = 1, Pass and Pass w/ Conditions = 0, This is the column the model will predict.

In [23]:
# Period column (same def as Task 1, with the 90-day grace period)
def get_period(date):
    if date < pd.Timestamp('2018-07-01'):
        return 'Pre'
    elif date < pd.Timestamp('2018-10-01'):
        return 'Grace (90 days)'
    elif date < pd.Timestamp('2019-07-01'):
        return 'Late transition'
    else:
        return 'Post'

df['Period'] = df['Inspection Date'].apply(get_period)
df['Period'].value_counts()

Period
Pre                64293
Post               47780
Late transition    11352
Grace (90 days)     3362
Name: count, dtype: int64

**Period.** Before the change, the 90-day grace period, the rest of the transition year, and after.

In [24]:
df.isnull().sum()

Inspection ID          0
DBA Name               0
AKA Name             327
License #              0
Facility Type          0
Risk                   0
Address                0
City                  46
State                 34
Zip                    1
Inspection Date        0
Inspection Type        1
Results                0
Violations         17816
Latitude             455
Longitude            455
Location             455
Year                   0
Fail                   0
Period                 0
dtype: int64

What still missing is: (AKA Name, City, State, Zip. Violations, Latitide, Longitude, Location ) and none of these columns will be used in the model, so you leave them as they are. Violations is empty when the inspection found nothing, which is normal.

In [25]:
df.to_csv('food_clean.csv', index=False)
print(df.shape)

(126787, 20)


We save the clean file. **Rows at the end:** (126787, 20). `food_clean.csv` is not uploaded to GitHub (see `.gitignore`), so each of us runs this notebook once on our own computer.